# 01 — Synthetic testbeds

This notebook runs the four experiments from Section 5 of the paper on two
small synthetic testbeds: an **exact synthetic gauge** (where the architecture
admits the gauge transformation by construction) and a **capacity-controlled
testbed** (where the feasible direction set is known and finite).

The goal is a fast correctness gate before the architecture-specific notebooks
(`02_attention`, `03_moe`, `04_gnn`). On Colab CPU this should run end-to-end
in a few minutes.

| Experiment | Tests                                                       | Testbed |
|------------|-------------------------------------------------------------|---------|
| 5.1        | Δ_gauge(c) ≈ 0; factor-only drift signature                 | Exact   |
| 5.2        | log m vs log λ slope; (1+2λ)m/‖g‖ ≈ const; cos(d, -g) ≈ 1   | Exact   |
| 5.3        | r_lin, r_quad, r_proj                                       | Both    |
| 5.4        | ρ-intervention sensitivity scales with m                    | Exact   |

All runs are saved via `RunLogger` under `<ROOT>/<testbed>/runs/`. The analysis
notebook (`05_analysis.ipynb`) loads everything by glob.


## Setup

In [ ]:
# Load the shared library in either local Jupyter or a Colab-hosted runtime.
import importlib
import importlib.util
import subprocess
import sys
from pathlib import Path

GITHUB_PACKAGE = "git+https://github.com/rblicht/Gauge.git"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def import_fresh_gauge_lib():
    sys.modules.pop("gauge_lib", None)
    return importlib.import_module("gauge_lib")


def install_gauge_lib_from_github():
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", GITHUB_PACKAGE])
    module = import_fresh_gauge_lib()
    print(f"Loaded gauge_lib from {Path(module.__file__).resolve()}")
    return module


def load_gauge_lib_from_file(gauge_lib_path: Path):
    gauge_lib_path = gauge_lib_path.resolve()
    sys.path.insert(0, str(gauge_lib_path.parent))
    sys.modules.pop("gauge_lib", None)
    spec = importlib.util.spec_from_file_location("gauge_lib", gauge_lib_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules["gauge_lib"] = module
    spec.loader.exec_module(module)
    print(f"Loaded gauge_lib from {gauge_lib_path}")
    return module

LOCAL_GAUGE_ROOT_CANDIDATES = [
    Path.cwd(),
    Path.home() / "Desktop" / "neurips",
    Path("/Users/ryanbruggeman/Desktop/neurips"),
]

if IN_COLAB:
    try:
        gauge_lib = install_gauge_lib_from_github()
    except Exception as exc:
        print(f"GitHub install failed: {exc}")
        print("If the repo is still private, make it public or upload gauge_lib.py as a temporary fallback.")
        from google.colab import files
        uploaded = files.upload()
        if "gauge_lib.py" not in uploaded:
            raise FileNotFoundError("Expected an uploaded file named gauge_lib.py")
        gauge_lib = load_gauge_lib_from_file(Path("/content/gauge_lib.py"))
else:
    gauge_lib_path = next(
        (root / "gauge_lib.py" for root in LOCAL_GAUGE_ROOT_CANDIDATES if (root / "gauge_lib.py").exists()),
        None,
    )
    if gauge_lib_path:
        gauge_lib = load_gauge_lib_from_file(gauge_lib_path)
    else:
        gauge_lib = import_fresh_gauge_lib()
        print(f"Loaded gauge_lib from {Path(gauge_lib.__file__).resolve()}")

In [ ]:
import math
import time
from copy import deepcopy
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from gauge_lib import (
    ALL_MODES,
    Controller, ControllerInfo,
    RunLogger, RunMeta,
    apply_gauge_traversal, delta_gauge,
    cos_d_neg_g, lambda_m_over_g,
    predict_e_lin, predict_e_proj,
    residual_norms, stationary_residual,
    set_seed,
)

print('gauge_lib modes:', ALL_MODES)

In [ ]:
# Detect Colab vs local; mount Drive if available.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/neurips2026')
except ImportError:
    IN_COLAB = False
    ROOT = Path.cwd() / 'neurips2026_local'

ROOT.mkdir(parents=True, exist_ok=True)
print(f'IN_COLAB = {IN_COLAB}')
print(f'ROOT     = {ROOT}')

In [ ]:
# Hyperparameters. Adjust if you want a faster smoke test or a fuller sweep.
SEEDS    = [0, 1, 2, 3, 4]
LAM_5_1  = 1e-2
LAM_5_4  = 1e-2
LAM_5_3C = 1e-2
LAMBDAS_5_2 = (1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2,
               1e-1, 3e-1, 1.0, 3.0, 10.0, 30.0)
C_VALUES_5_1 = (1e-2, 1e-1, 1.0, 1e1, 1e2)
RHOS_5_4 = (0.0, 0.25, 0.5, 1.0, 1.5, 2.0)

D_IN, N = 4, 4
K_CAP   = 4

# Per-experiment training step counts (Adam lr=5e-2 throughout).
STEPS_5_1   = 400
STEPS_5_2   = 1500
STEPS_5_4   = 600
STEPS_5_3C  = 1500

## Helpers

The same `ScalarHead`, training loop, and eval routine are used by both
testbeds. The exponential parameterization in `ScalarHead` is what makes the
synthetic gauge *exact*: scaling the output by a factor is exactly a bias
shift, so `apply_gauge_traversal` matches the (a, z) → (c⁻¹a, c·z) map without
floating-point slop.

In [ ]:
class ScalarHead(nn.Module):
    """Outputs a positive scalar via ``exp(w · x + b)``.

    The exponential form lets ``scale_output(factor)`` be implemented exactly
    as a bias shift, which is what gives the synthetic testbed exact gauge
    invariance under (a, z) → (c⁻¹a, c·z).
    """
    def __init__(self, d_in: int, init_log_scale: float = 0.0):
        super().__init__()
        self.linear = nn.Linear(d_in, 1)
        with torch.no_grad():
            self.linear.weight.mul_(0.1)
            self.linear.bias.fill_(init_log_scale)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.exp(self.linear(x).squeeze(-1))

    @torch.no_grad()
    def scale_output(self, factor: float) -> None:
        if factor <= 0:
            raise ValueError(f"factor must be positive, got {factor}")
        self.linear.bias.add_(math.log(factor))

In [ ]:
def task_loss(e: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
    """L_task = (1/2) ||e - t||^2 averaged over the batch."""
    return 0.5 * (e - t).pow(2).sum(dim=-1).mean()


def train_loop(model: nn.Module, data: Dict[str, torch.Tensor],
               steps: int, lr: float = 5e-2, log_every: int = 100,
               capacity: bool = False) -> List[Dict]:
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    log = []
    x, t = data["x_train"], data["t_train"]
    for step in range(steps):
        if capacity:
            e, info, _ = model(x, hard=False)
        else:
            e, info = model(x)
        L = task_loss(e, t) + info.penalty
        opt.zero_grad()
        L.backward()
        opt.step()
        if step % log_every == 0 or step == steps - 1:
            with torch.no_grad():
                log.append({
                    "step": step, "loss": L.item(),
                    "task_loss": task_loss(e, t).item(),
                    "penalty": info.penalty.item(),
                    "m_mean": info.m.mean().item(),
                    "z_norm_mean": info.z_norm.mean().item(),
                    "e_norm_mean": info.e_norm.mean().item(),
                })
    return log

In [ ]:
def eval_diagnostics_exact(model, data, lam):
    """Per-example diagnostics on the held-out set.

    For our quadratic loss L = ||e - t||²/2, we have g(0) = -t and H = I, so
    the linearized and curvature-corrected predictions are closed-form:
        e_lin  = -g/(2λ) = t/(2λ)
        e_quad = t/(1+2λ)
    Computing them this way avoids per-example HVP/CG.
    """
    model.eval()
    x, t = data["x_eval"], data["t_eval"]
    with torch.no_grad():
        e_obs, info = model(x)
    g = -t                          # gradient of task loss at e=0
    g_obs = e_obs - t               # gradient at observed e

    e_lin  = predict_e_lin(g, lam)
    e_quad = -1.0 / (1.0 + 2.0 * lam) * g

    res = residual_norms(e_obs, e_lin=e_lin, e_quad=e_quad)
    r_stat = stationary_residual(g_obs, e_obs, lam)

    return {
        "a": info.a.numpy(), "z_norm": info.z_norm.numpy(),
        "e_norm": info.e_norm.numpy(), "m": info.m.numpy(),
        "g_norm": g.norm(dim=-1).numpy(),
        "lambda_m_over_g": lambda_m_over_g(info.m, g, lam).numpy(),
        "cos_d_neg_g": cos_d_neg_g(info.d, g).numpy(),
        "r_lin": res["r_lin"].numpy(),
        "r_quad": res["r_quad"].numpy(),
        "r_stat": r_stat.numpy(),
        "e_obs": e_obs.numpy(), "t": t.numpy(),
        "task_metric": task_loss(e_obs, t).item(),
    }

In [ ]:
def make_data_exact(d_in: int = D_IN, n: int = N, n_train: int = 1000,
                    n_eval: int = 256, noise_std: float = 0.05, seed: int = 0):
    """t = T x + noise, T scaled by 1/sqrt(d_in)."""
    g = torch.Generator().manual_seed(seed)
    T = torch.randn(n, d_in, generator=g) / math.sqrt(d_in)
    x_train = torch.randn(n_train, d_in, generator=g)
    x_eval  = torch.randn(n_eval,  d_in, generator=g)
    t_train = x_train @ T.T + noise_std * torch.randn(n_train, n, generator=g)
    t_eval  = x_eval  @ T.T + noise_std * torch.randn(n_eval,  n, generator=g)
    return {"T": T, "x_train": x_train, "x_eval": x_eval,
            "t_train": t_train, "t_eval": t_eval}


def make_data_capacity(d_in: int = D_IN, n: int = N, n_train: int = 1000,
                       n_eval: int = 256, noise_std: float = 0.05, seed: int = 0):
    """Same form as `make_data_exact`. Targets are not generally aligned with
    any single feasible direction, so the optimal feasible response requires
    picking the best d_k and projecting."""
    g = torch.Generator().manual_seed(seed)
    T = torch.randn(n, d_in, generator=g) / math.sqrt(d_in)
    x_train = torch.randn(n_train, d_in, generator=g)
    x_eval  = torch.randn(n_eval,  d_in, generator=g)
    t_train = x_train @ T.T + noise_std * torch.randn(n_train, n, generator=g)
    t_eval  = x_eval  @ T.T + noise_std * torch.randn(n_eval,  n, generator=g)
    return {"T": T, "x_train": x_train, "x_eval": x_eval,
            "t_train": t_train, "t_eval": t_eval}

## Testbed 1 — Exact synthetic gauge

```
z = W x        (linear aggregator, no bias)
s = exp(w · x + b)     (positive scalar via ScalarHead)
e, info = Controller(mode, λ)(z, s)
```

The (a, z) → (c⁻¹a, c·z) gauge is realized exactly by parameter interventions:
- `scale_amp(c)`: shifts ScalarHead's bias by `log(c)` (output multiplied by c).
- `scale_agg(c)`: multiplies the aggregator weight matrix by c.

For `gauge_fixed`, the scalar IS m (the gauge-invariant magnitude), so
`scale_amp` is a no-op — m doesn't change under (a, z) gauge transformations
since m = ‖e‖ is observable.

In [ ]:
class ExactSyntheticModel(nn.Module):
    def __init__(self, d_in: int, n: int, mode: str, lam: float,
                 fixed_magnitude: float = 1.0):
        super().__init__()
        self.aggregator = nn.Linear(d_in, n, bias=False)
        self.scalar_head = ScalarHead(d_in)
        self.controller = Controller(mode=mode, lam=lam,
                                     fixed_magnitude=fixed_magnitude)
        self.mode = mode
        self.lam = lam

    def forward(self, x):
        z = self.aggregator(x)         # (B, n)
        s = self.scalar_head(x)        # (B,)
        return self.controller(z, s)

    def scale_amp(self, factor: float) -> None:
        # gauge_fixed: scalar is m (gauge-invariant)  → no-op
        # dir_norm:    scalar is unused              → no-op
        if self.controller.mode in ("gauge_fixed", "dir_norm"):
            return
        self.scalar_head.scale_output(factor)

    @torch.no_grad()
    def scale_agg(self, factor: float) -> None:
        self.aggregator.weight.mul_(factor)

### Experiment 5.1 — Gauge traversal and factor-only drift

Train all six controller variants at fixed λ. After training, apply the gauge
intervention `(a, z) → (c⁻¹a, c·z)` for `c ∈ {1e-2, 1e-1, 1, 1e1, 1e2}` and
measure the relative deviation of the realized displacement.

**Prediction**: Δ_gauge(c) ≈ 0 across all variants in this *exact* gauge —
the architecture admits the transformation by construction. The
non-trivial finding (covered in 05_analysis) is that with factor-only
penalties (`amp_only`, `agg_only`), training drifts along the gauge: a → 0
while ‖z‖ → ∞ (or vice-versa) with ‖e‖ stable.

In [ ]:
def run_experiment_5_1(root: Path, lam: float, seeds: List[int], c_values, steps: int):
    print(f"=== 5.1 gauge traversal (lam={lam}) ===")
    t0 = time.time()
    for seed in seeds:
        for variant in ALL_MODES:
            set_seed(seed)
            data = make_data_exact(seed=seed)
            model = ExactSyntheticModel(D_IN, N, mode=variant, lam=lam)
            train_log = train_loop(model, data, steps=steps, log_every=20)

            # Post-training gauge traversal
            with torch.no_grad():
                e_before, _ = model(data["x_eval"])

            traversal = {}
            for c in c_values:
                m_copy = deepcopy(model)
                apply_gauge_traversal(c, m_copy.scale_amp, m_copy.scale_agg)
                with torch.no_grad():
                    e_after, info_after = m_copy(data["x_eval"])
                rel = (e_after - e_before).norm(dim=-1) / e_before.norm(dim=-1).clamp_min(1e-12)
                traversal[float(c)] = {
                    "delta_gauge_mean": rel.mean().item(),
                    "delta_gauge_max":  rel.max().item(),
                    "a_mean":      info_after.a.mean().item(),
                    "z_norm_mean": info_after.z_norm.mean().item(),
                    "e_norm_mean": info_after.e_norm.mean().item(),
                }

            diag = eval_diagnostics_exact(model, data, lam)
            meta = RunMeta(testbed="synthetic_exact_5_1", variant=variant,
                           lam=lam, seed=seed)
            logger = RunLogger(root, meta, config={
                "experiment": "5_1", "lam": lam, "seed": seed,
                "d_in": D_IN, "n": N, "steps": steps,
            })
            for row in train_log:
                logger.log_step(**row)
            logger.update_final(**diag)
            logger.set_final("gauge_traversal", traversal)
            logger.save()
        print(f"  seed={seed} done")
    print(f"  total {time.time() - t0:.1f}s")


run_experiment_5_1(ROOT, LAM_5_1, SEEDS, C_VALUES_5_1, STEPS_5_1)

**Sanity-check plot**: max-Δ_gauge across c, by variant. All bars should be tiny (≲ 1e-6).

In [ ]:
# 5.1 sanity check
runs = RunLogger.load_testbed(ROOT, "synthetic_exact_5_1")
delta_max_by_variant = defaultdict(list)
for r in runs:
    v = r["meta"]["variant"]
    for c, stats in r["final"]["gauge_traversal"].items():
        delta_max_by_variant[v].append(stats["delta_gauge_max"])

variants_order = list(ALL_MODES)
max_deltas = [max(delta_max_by_variant[v]) for v in variants_order]

fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(range(len(variants_order)), max_deltas, color="steelblue")
ax.set_yscale("log")
ax.set_xticks(range(len(variants_order)))
ax.set_xticklabels(variants_order, rotation=15)
ax.set_ylabel("max Δ_gauge across c, seeds")
ax.axhline(1e-6, color="red", linestyle="--", linewidth=1, label="numerical floor")
ax.set_title("5.1 — gauge traversal residuals (exact synthetic)")
ax.legend()
plt.tight_layout()
plt.show()

print("\nWorst-case Δ_gauge per variant:")
for v, d in zip(variants_order, max_deltas):
    print(f"  {v:>14}: {d:.2e}")

### Experiment 5.2 — Controller calibration under λ sweeps

λ-sweep for the gauge-fixed controller. We log `m, ‖g‖, cos(d, -g)` per
held-out example and check three things:

1. **Slope** of `log10 m` vs `log10 λ` → -1 in the asymptotic regime.
   For our quadratic task with `H = I` the controlled equilibrium is
   `m* = ‖t‖/(1+2λ)`, giving slope `-2λ/(1+2λ)`. This approaches -1 only
   asymptotically, so we fit only over `λ ≥ 1`.

2. **Calibration ratio** `(1+2λ)·m / ‖g‖` ≈ const across λ — the cleaner
   signature for our task. (`2λm/‖g‖` from Corollary 1 only equals 1 in
   the linearized limit.)

3. **Direction alignment** `cos(d, -g) ≈ 1` for the gauge-fixed controller.

In [ ]:
def run_experiment_5_2(root: Path, lambdas, seeds: List[int], steps: int):
    print(f"=== 5.2 lambda sweep (gauge_fixed) ===")
    t0 = time.time()
    for seed in seeds:
        for lam in lambdas:
            set_seed(seed)
            data = make_data_exact(seed=seed)
            model = ExactSyntheticModel(D_IN, N, mode="gauge_fixed", lam=lam)
            train_log = train_loop(model, data, steps=steps, log_every=200)
            diag = eval_diagnostics_exact(model, data, lam)

            meta = RunMeta(testbed="synthetic_exact_5_2", variant="gauge_fixed",
                           lam=lam, seed=seed)
            logger = RunLogger(root, meta, config={
                "experiment": "5_2", "lam": lam, "seed": seed, "steps": steps,
            })
            for row in train_log:
                logger.log_step(**row)
            logger.update_final(**diag)
            logger.save()
        print(f"  seed={seed} done")
    print(f"  total {time.time() - t0:.1f}s")


run_experiment_5_2(ROOT, LAMBDAS_5_2, SEEDS, STEPS_5_2)

**Sanity-check plot**: log m vs log λ, calibration ratio, and direction alignment.

In [ ]:
# 5.2 sanity check
runs = RunLogger.load_testbed(ROOT, "synthetic_exact_5_2")
m_by_lam   = defaultdict(list)
ratio_lin  = defaultdict(list)
ratio_eq   = defaultdict(list)
cos_by_lam = defaultdict(list)

for r in runs:
    lam = r["meta"]["lam"]
    m   = r["final"]["m"]
    g   = r["final"]["g_norm"]
    m_by_lam[lam].extend(m.tolist())
    ratio_lin[lam].extend((2 * lam * m / g).tolist())
    ratio_eq[lam].extend(((1 + 2 * lam) * m / g).tolist())
    cos_by_lam[lam].extend(r["final"]["cos_d_neg_g"].tolist())

lams_sorted = sorted(m_by_lam.keys())
log_lam = np.log10(np.array(lams_sorted))
log_m_med = np.log10(np.array([np.median(m_by_lam[l]) for l in lams_sorted]))

# Asymptotic slope (λ ≥ 1)
mask = np.array(lams_sorted) >= 1.0
slope_asymp, _ = (np.polyfit(log_lam[mask], log_m_med[mask], 1)
                  if mask.sum() >= 2 else (np.nan, 0))

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))

# (a) log m vs log λ
axes[0].plot(log_lam, log_m_med, "o-", color="steelblue", label="median m")
axes[0].plot(log_lam[mask], log_m_med[mask], "s", color="darkorange",
             label=f"asymp fit slope={slope_asymp:.2f}")
axes[0].set_xlabel("log10 λ"); axes[0].set_ylabel("log10 m")
axes[0].set_title("5.2 — m vs λ")
axes[0].legend()

# (b) calibration ratios
med_lin = [np.median(ratio_lin[l]) for l in lams_sorted]
med_eq  = [np.median(ratio_eq[l])  for l in lams_sorted]
axes[1].plot(lams_sorted, med_lin, "o-", label="2λm/‖g‖ (Cor. 1)")
axes[1].plot(lams_sorted, med_eq,  "s-", label="(1+2λ)m/‖g‖ (full eq.)")
axes[1].axhline(1.0, color="grey", linestyle="--", linewidth=1)
axes[1].set_xscale("log")
axes[1].set_xlabel("λ"); axes[1].set_ylabel("ratio")
axes[1].set_title("calibration")
axes[1].legend()

# (c) direction alignment
med_cos = [np.median(cos_by_lam[l]) for l in lams_sorted]
axes[2].plot(lams_sorted, med_cos, "o-", color="green")
axes[2].axhline(1.0, color="grey", linestyle="--", linewidth=1)
axes[2].set_xscale("log")
axes[2].set_xlabel("λ"); axes[2].set_ylabel("cos(d, -g)")
axes[2].set_title("direction alignment")
axes[2].set_ylim(0.9, 1.01)

plt.tight_layout()
plt.show()

print(f"\nAsymptotic slope (λ ≥ 1): {slope_asymp:.3f}  (theoretical limit: -1)")
print(f"(1+2λ)m/‖g‖ at λ=1e-2: {np.median(ratio_eq[1e-2]):.3f}  (should be ≈ const across λ)")
print(f"Median cos(d, -g):     {np.median(med_cos):.4f}  (should be ≈ 1)")

### Experiment 5.3 — Residual decomposition (exact synthetic)

The 5.2 runs already include `r_lin` and `r_quad` per held-out example, plus
the stationary residual `r_stat`. Here we just summarize the ratio
`r_quad / r_lin` across λ:

- For our quadratic task `H = I`, `e_quad = t/(1+2λ)` is *exact*, so
  `r_quad → 0` and the ratio collapses to ~0 at small λ where curvature
  matters most.
- At large λ, `2λI` dominates `H` and the linearized prediction itself
  becomes accurate, so `r_lin` shrinks too and the ratio approaches 1.

The capacity-projection residual `r_proj` is tested in Testbed 2 below
(the exact testbed has no capacity constraint).

In [ ]:
# 5.3 — residual summary from 5.2 runs
runs = RunLogger.load_testbed(ROOT, "synthetic_exact_5_2")
r_lin_by_lam, r_quad_by_lam, r_stat_by_lam = (defaultdict(list) for _ in range(3))
for r in runs:
    lam = r["meta"]["lam"]
    r_lin_by_lam[lam].extend(r["final"]["r_lin"].tolist())
    r_quad_by_lam[lam].extend(r["final"]["r_quad"].tolist())
    r_stat_by_lam[lam].extend(r["final"]["r_stat"].tolist())

lams = sorted(r_lin_by_lam.keys())
med_lin  = np.array([np.median(r_lin_by_lam[l])  for l in lams])
med_quad = np.array([np.median(r_quad_by_lam[l]) for l in lams])
med_stat = np.array([np.median(r_stat_by_lam[l]) for l in lams])

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

axes[0].loglog(lams, med_lin,  "o-", label="r_lin")
axes[0].loglog(lams, med_quad, "s-", label="r_quad")
axes[0].loglog(lams, med_stat, "^-", label="r_stat", alpha=0.7)
axes[0].set_xlabel("λ"); axes[0].set_ylabel("residual norm (median)")
axes[0].set_title("5.3 — residual norms vs λ")
axes[0].legend()

axes[1].semilogx(lams, med_quad / med_lin, "o-", color="purple")
axes[1].axhline(1.0, color="grey", linestyle="--", linewidth=1)
axes[1].set_xlabel("λ"); axes[1].set_ylabel("r_quad / r_lin")
axes[1].set_title("curvature-corrected residual ratio")
axes[1].set_ylim(-0.05, 1.1)

plt.tight_layout()
plt.show()

print(f"\nr_quad / r_lin at λ=1e-3:  {med_quad[lams.index(1e-3)] / med_lin[lams.index(1e-3)]:.2e}")
print(f"r_quad / r_lin at λ=10:    {med_quad[lams.index(10.0)] / med_lin[lams.index(10.0)]:.2e}")
print("(Curvature correction is exact for our quadratic task; ratio ~0 at small λ.)")

### Experiment 5.4 — Magnitude calibration and causal intervention

Train gauge-fixed at fixed λ, then at evaluation replace the realized
displacement `e = m·d` with `e_ρ = ρ·m·d` for `ρ ∈ {0, 0.25, 0.5, 1, 1.5, 2}`.

**Prediction**: examples assigned larger `m` should be more sensitive to
shrinkage (`ρ → 0`) — sensitivity scales with `m`. Overscaling (`ρ > 1`)
need not improve performance because `m` is already at the controlled
equilibrium.

We bin held-out examples into low/medium/high m terciles and compare the
sensitivity `loss(ρ=0) − loss(ρ=1)` across bins.

In [ ]:
def run_experiment_5_4(root: Path, lam: float, seeds: List[int], rhos, steps: int):
    print(f"=== 5.4 rho intervention (lam={lam}) ===")
    t0 = time.time()
    for seed in seeds:
        set_seed(seed)
        data = make_data_exact(seed=seed)
        model = ExactSyntheticModel(D_IN, N, mode="gauge_fixed", lam=lam)
        train_log = train_loop(model, data, steps=steps)
        diag = eval_diagnostics_exact(model, data, lam)

        x_eval, t_eval = data["x_eval"], data["t_eval"]
        with torch.no_grad():
            e_obs, info = model(x_eval)
            m, d = info.m, info.d

        rho_results = {}
        for rho in rhos:
            with torch.no_grad():
                e_rho = (rho * m).unsqueeze(-1) * d
            per_ex = 0.5 * (e_rho - t_eval).pow(2).sum(dim=-1)
            rho_results[float(rho)] = {
                "task_metric_mean": per_ex.mean().item(),
                "per_example_loss": per_ex.numpy(),
            }

        m_np = m.numpy()
        q33, q66 = np.quantile(m_np, [1/3, 2/3])
        bins = np.where(m_np <= q33, 0, np.where(m_np <= q66, 1, 2))

        meta = RunMeta(testbed="synthetic_exact_5_4", variant="gauge_fixed",
                       lam=lam, seed=seed)
        logger = RunLogger(root, meta, config={
            "experiment": "5_4", "lam": lam, "seed": seed, "steps": steps,
        })
        for row in train_log:
            logger.log_step(**row)
        logger.update_final(**diag)
        logger.set_final("rho_intervention", rho_results)
        logger.set_final("m_bin", bins)
        logger.save()
    print(f"  total {time.time() - t0:.1f}s")


run_experiment_5_4(ROOT, LAM_5_4, SEEDS, RHOS_5_4, STEPS_5_4)

**Sanity-check plot**: task loss vs ρ, separated by m-tercile.

In [ ]:
# 5.4 sanity check
runs = RunLogger.load_testbed(ROOT, "synthetic_exact_5_4")
loss_by_bin = defaultdict(lambda: defaultdict(list))
for r in runs:
    bins = r["final"]["m_bin"]
    for rho_str, stats in r["final"]["rho_intervention"].items():
        rho = float(rho_str)
        per = np.asarray(stats["per_example_loss"])
        for b in [0, 1, 2]:
            mask = bins == b
            if mask.any():
                loss_by_bin[b][rho].extend(per[mask].tolist())

fig, ax = plt.subplots(figsize=(6, 3.5))
labels = ["low-m", "med-m", "high-m"]
colors = ["steelblue", "darkorange", "firebrick"]
for b, label, color in zip([0, 1, 2], labels, colors):
    rhos = sorted(loss_by_bin[b].keys())
    means = [np.mean(loss_by_bin[b][r]) for r in rhos]
    ax.plot(rhos, means, "o-", color=color, label=label)
ax.axvline(1.0, color="grey", linestyle="--", linewidth=1, alpha=0.7)
ax.set_xlabel("ρ"); ax.set_ylabel("task loss (mean)")
ax.set_title("5.4 — ρ intervention by m-tercile")
ax.legend()
plt.tight_layout()
plt.show()

print("\nSensitivity to shrinkage (loss[ρ=0] - loss[ρ=1]) by m-tercile:")
for b, label in zip([0, 1, 2], labels):
    sens = np.mean(loss_by_bin[b][0.0]) - np.mean(loss_by_bin[b][1.0])
    print(f"  {label:>6}: {sens:.3f}")
print("\nExpected ordering: high-m has the largest sensitivity.")

## Testbed 2 — Capacity-controlled

K fixed unit directions `{d_1, ..., d_K}` (sampled once per seed and
orthonormalized when `K ≤ n`). The model picks among them via gating
logits `MLP(x)`. During training we use a soft mixture with low temperature
(τ=0.1) for differentiability; at evaluation we use hard `argmax` so the
realized direction lies on the discrete feasible set.

This setup gives us a *known* feasible direction set, which is what
Corollary 3 needs.

In [ ]:
class CapacityControlledModel(nn.Module):
    def __init__(self, d_in: int, n: int, K: int, mode: str, lam: float,
                 tau: float = 0.1, fixed_magnitude: float = 1.0, seed: int = 0):
        super().__init__()
        g = torch.Generator().manual_seed(seed)
        D = torch.randn(K, n, generator=g)
        if K <= n:
            Q, _ = torch.linalg.qr(D.T)   # orthonormal columns
            D = Q.T
        else:
            D = D / D.norm(dim=-1, keepdim=True)
        self.register_buffer("D", D)
        self.gate_logits = nn.Linear(d_in, K)
        self.scalar_head = ScalarHead(d_in)
        self.controller = Controller(mode=mode, lam=lam, fixed_magnitude=fixed_magnitude)
        self.tau = tau
        self.K, self.n = K, n

    def forward(self, x, hard: bool = False):
        logits = self.gate_logits(x)
        if hard:
            idx = logits.argmax(dim=-1)
            z = self.D[idx]                       # (B, n), ‖z‖=1
        else:
            alpha = F.softmax(logits / self.tau, dim=-1)
            z = alpha @ self.D                    # (B, n)
        s = self.scalar_head(x)
        e, info = self.controller(z, s)
        return e, info, logits

### Experiment 5.3 (capacity) — Direction selection and r_proj

Train gauge-fixed on the capacity-controlled testbed; evaluate in *hard*
mode and compare to Corollary 3's prediction:

- The model should pick `d* = argmax_k [-⟨g, d_k⟩]_+`.
- Magnitude: `m* = [-⟨g, d*⟩]_+ / (2λ)`.
- The capacity-projected prediction `e_proj = m* · d*` should be a closer
  fit to the model's output than the unconstrained `e_lin = -g/(2λ)`.

In [ ]:
def best_feasible_direction(g: torch.Tensor, D: torch.Tensor):
    """argmax_k [-g·d_k]_+ and the corresponding [-g·d_k]_+."""
    inner = -g @ D.T                              # (B, K)
    relu = inner.clamp_min(0.0)
    best_idx = relu.argmax(dim=-1)                # (B,)
    best_neg_inner = relu.gather(-1, best_idx.unsqueeze(-1)).squeeze(-1)
    return best_idx, best_neg_inner


def run_experiment_5_3_capacity(root: Path, lam: float, seeds: List[int],
                                K: int, n: int, d_in: int, steps: int):
    print(f"=== 5.3 capacity (lam={lam}, K={K}, n={n}) ===")
    t0 = time.time()
    for seed in seeds:
        set_seed(seed)
        data = make_data_capacity(d_in=d_in, n=n, seed=seed)
        model = CapacityControlledModel(d_in=d_in, n=n, K=K, mode="gauge_fixed",
                                        lam=lam, seed=seed)
        train_log = train_loop(model, data, steps=steps, capacity=True)

        x_eval, t_eval = data["x_eval"], data["t_eval"]
        with torch.no_grad():
            e_obs, info, logits = model(x_eval, hard=True)
        g = -t_eval
        e_lin = predict_e_lin(g, lam)
        D = model.D
        best_idx, best_neg_inner = best_feasible_direction(g, D)
        m_pred = best_neg_inner / (2.0 * lam)
        e_proj_pred = m_pred.unsqueeze(-1) * D[best_idx]

        sel_idx = logits.argmax(dim=-1)
        cos_obs_pred = (D[sel_idx] * D[best_idx]).sum(dim=-1)
        match_rate = (sel_idx == best_idx).float().mean().item()

        res = residual_norms(e_obs, e_lin=e_lin, e_proj=e_proj_pred)
        diag = {
            "a": info.a.numpy(), "z_norm": info.z_norm.numpy(),
            "e_norm": info.e_norm.numpy(), "m": info.m.numpy(),
            "g_norm": g.norm(dim=-1).numpy(),
            "cos_d_neg_g": cos_d_neg_g(info.d, g).numpy(),
            "r_lin": res["r_lin"].numpy(),
            "r_proj": res["r_proj"].numpy(),
            "match_rate": match_rate,
            "best_idx": best_idx.numpy(),
            "sel_idx": sel_idx.numpy(),
            "cos_obs_pred_dir": cos_obs_pred.numpy(),
            "task_metric": 0.5 * ((e_obs - t_eval) ** 2).sum(dim=-1).mean().item(),
        }
        meta = RunMeta(testbed="synthetic_capacity_5_3", variant="gauge_fixed",
                       lam=lam, seed=seed)
        logger = RunLogger(root, meta, config={
            "experiment": "5_3_cap", "lam": lam, "seed": seed,
            "K": K, "n": n, "d_in": d_in, "steps": steps,
        })
        for row in train_log:
            logger.log_step(**row)
        logger.update_final(**diag)
        logger.save()
    print(f"  total {time.time() - t0:.1f}s")


run_experiment_5_3_capacity(ROOT, LAM_5_3C, SEEDS, K_CAP, N, D_IN, STEPS_5_3C)

**Sanity-check plot**: residual ratio, direction match rate, alignment with best feasible direction.

In [ ]:
# 5.3-capacity sanity check
runs = RunLogger.load_testbed(ROOT, "synthetic_capacity_5_3")
match_rates, rlin_all, rproj_all = [], [], []
cos_g_all, cos_pred_all = [], []
for r in runs:
    match_rates.append(r["final"]["match_rate"])
    rlin_all.extend(r["final"]["r_lin"].tolist())
    rproj_all.extend(r["final"]["r_proj"].tolist())
    cos_g_all.extend(r["final"]["cos_d_neg_g"].tolist())
    cos_pred_all.extend(r["final"]["cos_obs_pred_dir"].tolist())

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))

axes[0].bar(["r_lin", "r_proj"], [np.median(rlin_all), np.median(rproj_all)],
            color=["steelblue", "darkorange"])
axes[0].set_ylabel("median residual"); axes[0].set_title("5.3-cap — residual norms")

axes[1].hist(match_rates, bins=10, color="seagreen")
axes[1].set_xlim(0, 1.05); axes[1].set_xlabel("match rate (per seed)")
axes[1].set_title("direction selection match rate")

axes[2].hist(cos_pred_all, bins=20, alpha=0.6, label="cos(d_obs, best feasible)")
axes[2].hist(cos_g_all,    bins=20, alpha=0.6, label="cos(d_obs, -g/‖g‖)")
axes[2].set_xlabel("cosine"); axes[2].legend()
axes[2].set_title("direction alignment")

plt.tight_layout()
plt.show()

print(f"\nMatch rate (median):                {np.median(match_rates):.3f}")
print(f"r_proj / r_lin:                     {np.median(rproj_all) / np.median(rlin_all):.3f}  (< 1 confirms capacity prediction)")
print(f"cos(d_obs, best feasible) (median): {np.median(cos_pred_all):.3f}  (≈ 1)")
print(f"cos(d_obs, -g/‖g‖) (median):         {np.median(cos_g_all):.3f}  (limited by capacity)")

## Summary

A run-counts table you can sanity-check before moving on. The five `synthetic_*`
testbeds should each contain `runs/*.pkl` files matching the configuration
above. `05_analysis.ipynb` glob-loads all of them by testbed name.

In [ ]:
# Summary of saved runs
for testbed_dir in sorted(ROOT.iterdir()):
    if not testbed_dir.is_dir(): continue
    runs_dir = testbed_dir / "runs"
    if not runs_dir.exists(): continue
    files = sorted(runs_dir.glob("*.pkl"))
    print(f"{testbed_dir.name:<28} {len(files)} runs")
    # Show a few example filenames
    for f in files[:2]:
        print(f"    {f.name}")
    if len(files) > 2:
        print(f"    ... ({len(files) - 2} more)")